In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_PATH = '/content/drive/MyDrive/MoodSense_Paper2/'
os.makedirs(SAVE_PATH, exist_ok=True)
print(f'Save path ready: {SAVE_PATH}')

Mounted at /content/drive
Save path ready: /content/drive/MyDrive/MoodSense_Paper2/


In [2]:
!pip install -q transformers torch vaderSentiment textblob scikit-learn \
               datasets matplotlib seaborn pandas numpy scipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 2.0 MB/s eta 0:00:00


In [3]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
warnings.filterwarnings('ignore')

from scipy.stats import spearmanr
from sklearn.metrics import (
    mean_absolute_error, accuracy_score, classification_report,
    confusion_matrix, roc_auc_score, roc_curve,
    precision_score, recall_score, f1_score
)
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

import torch
from transformers import pipeline
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from textblob import TextBlob
from datasets import load_dataset

print('All imports successful.')
print(f'GPU available: {torch.cuda.is_available()}')
device = 0 if torch.cuda.is_available() else -1

All imports successful.
GPU available: False


In [4]:
df_raw = None

# Attempt 1
print('Trying DEPTWEET...')
try:
    ds = load_dataset("ontocord/CrisisMMD_humaid_deptweet", split="train")
    df_raw = ds.to_pandas()
    print(f'Loaded: {len(df_raw)} rows | Columns: {df_raw.columns.tolist()}')
except Exception as e:
    print(f'Failed: {e}')

# Attempt 2
if df_raw is None:
    print('Trying depression-reddit-cleaned...')
    try:
        ds = load_dataset("mrjunos/depression-reddit-cleaned", split="train")
        df_raw = ds.to_pandas()
        print(f'Loaded: {len(df_raw)} rows | Columns: {df_raw.columns.tolist()}')
    except Exception as e:
        print(f'Failed: {e}')

# Attempt 3
if df_raw is None:
    print('Trying ShreyaR/DepressionDetection...')
    try:
        ds = load_dataset("ShreyaR/DepressionDetection", split="train")
        df_raw = ds.to_pandas()
        print(f'Loaded: {len(df_raw)} rows | Columns: {df_raw.columns.tolist()}')
    except Exception as e:
        print(f'Failed: {e}')

# Attempt 4
if df_raw is None:
    print('Trying dair-ai/emotion...')
    try:
        ds = load_dataset("dair-ai/emotion", split="train")
        df_raw = ds.to_pandas()
        print(f'Loaded: {len(df_raw)} rows | Columns: {df_raw.columns.tolist()}')
    except Exception as e:
        print(f'Failed: {e}')

# Attempt 5 — guaranteed fallback
if df_raw is None:
    print('Using guaranteed fallback: twitter-financial-news-sentiment...')
    try:
        ds = load_dataset("zeroshot/twitter-financial-news-sentiment", split="train")
        df_raw = ds.to_pandas()
        print(f'Loaded: {len(df_raw)} rows | Columns: {df_raw.columns.tolist()}')
    except Exception as e:
        print(f'Failed: {e}')

print('\nFinal columns:', df_raw.columns.tolist())
print('\nFirst 3 rows:')
print(df_raw.head(3))
print('\nAll unique label values:')
for col in df_raw.columns:
    print(f'  {col}: {df_raw[col].unique()[:10]}')

Trying DEPTWEET...


Failed: Dataset 'ontocord/CrisisMMD_humaid_deptweet' doesn't exist on the Hub or cannot be accessed.
Trying depression-reddit-cleaned...


Failed: Dataset scripts are no longer supported, but found depression-reddit-cleaned.py
Trying ShreyaR/DepressionDetection...


Loaded: 7731 rows | Columns: ['clean_text', 'is_depression']

Final columns: ['clean_text', 'is_depression']

First 3 rows:
                                          clean_text  is_depression
0  we understand that most people who reply immed...              1
1  welcome to r depression s check in post a plac...              1
2  anyone else instead of sleeping more when depr...              1

All unique label values:
  clean_text: ['we understand that most people who reply immediately to an op with an invitation to talk privately mean only to help but this type of response usually lead to either disappointment or disaster it usually work out quite differently here than when you say pm me anytime in a casual social context we have huge admiration and appreciation for the goodwill and good citizenship of so many of you who support others here and flag inappropriate content even more so because we know that so many of you are struggling yourselves we re hard at work behind the scene on m

In [5]:
df = df_raw.copy()

# ── Auto-find text column ──────────────────────────────────────────────────
text_candidates = ['text', 'tweet', 'post', 'message', 'content',
                   'sentence', 'comment', 'body', 'description']
TEXT_COL = next((c for c in text_candidates if c in df.columns), None)
if TEXT_COL is None:
    TEXT_COL = df.select_dtypes(include='object').columns[0]

# ── Auto-find label column ─────────────────────────────────────────────────
label_candidates = ['label', 'severity', 'class', 'depression',
                    'target', 'category', 'level', 'score', 'sentiment']
LABEL_COL = next((c for c in label_candidates if c in df.columns), None)
if LABEL_COL is None:
    LABEL_COL = [c for c in df.columns if c != TEXT_COL][0]

print(f'TEXT_COL  = {TEXT_COL}')
print(f'LABEL_COL = {LABEL_COL}')

# ── Rename ─────────────────────────────────────────────────────────────────
df = df[[TEXT_COL, LABEL_COL]].copy()
df.columns = ['text', 'label_raw']
df['text'] = df['text'].astype(str).str.strip()
df = df[df['text'].str.len() > 10].copy()

# ── Handle string labels ───────────────────────────────────────────────────
unique_raw = df['label_raw'].unique()
print(f'\nRaw label values: {unique_raw[:10]}')

# If labels are strings, convert to integers
if df['label_raw'].dtype == object:
    label_list = sorted(df['label_raw'].dropna().unique().tolist())
    label2int  = {l: i for i, l in enumerate(label_list)}
    print(f'String label mapping: {label2int}')
    df['severity_raw'] = df['label_raw'].map(label2int)
else:
    df['severity_raw'] = pd.to_numeric(df['label_raw'], errors='coerce')

df = df.dropna(subset=['severity_raw'])
df['severity_raw'] = df['severity_raw'].astype(int)

# ── Auto PHQ-9 mapping ─────────────────────────────────────────────────────
unique_labels = sorted(df['severity_raw'].unique())
print(f'\nNumeric labels: {unique_labels}')

if set(unique_labels) <= {0, 1}:
    print('Binary dataset: 0=healthy, 1=depressed')
    phq_map = {0: 2, 1: 15}
elif max(unique_labels) == 2:
    print('3-class dataset')
    phq_map = {0: 2, 1: 12, 2: 20}
elif max(unique_labels) == 3:
    print('4-class dataset')
    phq_map = {0: 2, 1: 7, 2: 12, 3: 20}
else:
    print('Custom range — linear mapping')
    phq_map = {v: int(v / max(unique_labels) * 27) for v in unique_labels}

print(f'PHQ map: {phq_map}')
df['phq9_score'] = df['severity_raw'].map(phq_map)
df = df.dropna(subset=['phq9_score'])
df['phq9_score'] = df['phq9_score'].astype(int)

# ── Derived columns ────────────────────────────────────────────────────────
def phq9_to_3class(s):
    if s <= 4:  return 0
    if s <= 14: return 1
    return 2

def phq9_to_4class(s):
    if s <= 4:  return 0
    if s <= 9:  return 1
    if s <= 14: return 2
    return 3

df['severity_4']    = df['phq9_score'].apply(phq9_to_4class)
df['severity_3']    = df['phq9_score'].apply(phq9_to_3class)
df['at_risk']       = (df['phq9_score'] >= 10).astype(int)
df['phq9_wellness'] = 100.0 - (df['phq9_score'] / 27.0 * 100.0)

# ── Sample 5000 ────────────────────────────────────────────────────────────
df = df.sample(n=min(5000, len(df)), random_state=42).reset_index(drop=True)

# ── Validate both classes exist ────────────────────────────────────────────
assert df['at_risk'].nunique() == 2, \
    f'ERROR: at_risk has only 1 class. PHQ map may need adjustment. Scores: {df["phq9_score"].unique()}'

print(f'\nDataset ready: {len(df)} samples')
print('\nat_risk:')
print(df['at_risk'].value_counts().rename({0:'not at risk', 1:'at risk'}))
print('\nseverity_3:')
print(df['severity_3'].value_counts().sort_index()
      .rename({0:'minimal', 1:'moderate', 2:'severe'}))
print('\nphq9_score:')
print(df['phq9_score'].value_counts().sort_index())

TEXT_COL  = clean_text
LABEL_COL = is_depression

Raw label values: [1 0]

Numeric labels: [np.int64(0), np.int64(1)]
Binary dataset: 0=healthy, 1=depressed
PHQ map: {0: 2, 1: 15}

Dataset ready: 5000 samples

at_risk:
at_risk
not at risk    2524
at risk        2476
Name: count, dtype: int64

severity_3:
severity_3
minimal    2524
severe     2476
Name: count, dtype: int64

phq9_score:
phq9_score
2     2524
15    2476
Name: count, dtype: int64


In [6]:
print('Loading DistilBERT...')
bert_pipe = pipeline(
    'text-classification',
    model='distilbert-base-uncased-finetuned-sst-2-english',
    device=device,
    truncation=True,
    max_length=512
)

print('Loading Hartmann emotion model...')
emotion_pipe = pipeline(
    'text-classification',
    model='j-hartmann/emotion-english-distilroberta-base',
    top_k=None,
    device=device,
    truncation=True,
    max_length=512
)

vader = SentimentIntensityAnalyzer()
print('All models loaded.')

Loading DistilBERT...


Loading Hartmann emotion model...


RobertaForSequenceClassification LOAD REPORT from: j-hartmann/emotion-english-distilroberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


All models loaded.


In [7]:
BATCH = 32
texts = df['text'].tolist()
N     = len(texts)

# DistilBERT
print(f'Running DistilBERT on {N} samples...')
bert_scores = []
for i in range(0, N, BATCH):
    batch   = texts[i:i+BATCH]
    results = bert_pipe(batch)
    for r in results:
        s = r['score'] if r['label'] == 'POSITIVE' else 1.0 - r['score']
        bert_scores.append(s)
    if (i // BATCH) % 20 == 0:
        print(f'  {min(i+BATCH,N)}/{N}')
df['s_bert'] = bert_scores

# VADER
print('Running VADER...')
df['s_vader'] = df['text'].apply(
    lambda t: (vader.polarity_scores(t)['compound'] + 1) / 2
)

# TextBlob
print('Running TextBlob...')
df['s_blob'] = df['text'].apply(
    lambda t: (TextBlob(t).sentiment.polarity + 1) / 2
)

# Emotion model
print(f'Running emotion model on {N} samples...')
joy_scores = []
for i in range(0, N, BATCH):
    batch   = texts[i:i+BATCH]
    results = emotion_pipe(batch)
    for r in results:
        lm      = {x['label']: x['score'] for x in r}
        joy     = lm.get('joy', 0.0)
        sadness = lm.get('sadness', 0.0)
        fear    = lm.get('fear', 0.0)
        valence = joy / max(joy + sadness + fear, 1e-9)
        joy_scores.append(valence)
    if (i // BATCH) % 20 == 0:
        print(f'  {min(i+BATCH,N)}/{N}')
df['s_emotion'] = joy_scores

print('Done.')

Running DistilBERT on 5000 samples...
  32/5000
  672/5000
  1312/5000
  1952/5000
  2592/5000
  3232/5000
  3872/5000
  4512/5000
Running VADER...
Running TextBlob...
Running emotion model on 5000 samples...
  32/5000
  672/5000
  1312/5000
  1952/5000
  2592/5000
  3232/5000
  3872/5000
  4512/5000
Done.


In [18]:
W_BERT, W_VADER, W_BLOB = 0.60, 0.20, 0.20

def wellbeing_score(s_bert, s_vader, s_blob, w1=W_BERT, w2=W_VADER, w3=W_BLOB):
    return (s_bert * w1 + s_vader * w2 + s_blob * w3) * 100

df['wb_ensemble']   = wellbeing_score(df['s_bert'], df['s_vader'], df['s_blob'])
df['wb_bert_only']  = df['s_bert']  * 100
df['wb_vader_only'] = df['s_vader'] * 100
df['wb_blob_only']  = df['s_blob']  * 100

def wb_to_3class(wb):
    if wb >= 65: return 0
    if wb >= 35: return 1
    return 2

def wb_to_4class(wb):
    if wb >= 80: return 0
    if wb >= 65: return 1
    if wb >= 35: return 2
    return 3

df['pred_3class'] = df['wb_ensemble'].apply(wb_to_3class)
df['pred_4class'] = df['wb_ensemble'].apply(wb_to_4class)

print('Wellbeing Score computed.')
print(df[['text','phq9_score','wb_ensemble',
          'pred_3class','severity_3']].head(5).to_string(index=False))

Wellbeing Score computed.
                                                                                                                                                                        text  phq9_score  wb_ensemble  pred_3class  severity_3
                                                                     argh driving into london today made a wrong turn at king x stuck in an extra 0 minute of logjam traffic           2    11.436044            2           0
i smoked for the first time in month and now i m freaking out and idk what to do edit i m all good now blocked out the stress for a little bit now getting food with the boy          15    16.976646            2           2
                                                                                          marleyuk i think you spoke too soon big black rain cloud charging towards town now           2    21.561740            2           0
                                                                          i miss y

In [19]:
print('=== PAPER 1: Binary Risk Detection (LR + TF-IDF) ===')

X = df['text'].values
y = df['at_risk'].values

print(f'Class distribution: {dict(zip(*np.unique(y, return_counts=True)))}')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

tfidf      = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))
X_train_tf = tfidf.fit_transform(X_train)
X_test_tf  = tfidf.transform(X_test)

lr = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
lr.fit(X_train_tf, y_train)
y_pred_lr = lr.predict(X_test_tf)
y_prob_lr = lr.predict_proba(X_test_tf)[:, 1]

p1_acc  = accuracy_score(y_test, y_pred_lr)
p1_prec = precision_score(y_test, y_pred_lr, zero_division=0)
p1_rec  = recall_score(y_test, y_pred_lr, zero_division=0)
p1_f1   = f1_score(y_test, y_pred_lr, zero_division=0)
p1_auc  = roc_auc_score(y_test, y_prob_lr)

df_p1 = pd.DataFrame([{
    'Model':     'MoodSense (LR + TF-IDF)',
    'Accuracy':  round(p1_acc,  4),
    'Precision': round(p1_prec, 4),
    'Recall':    round(p1_rec,  4),
    'F1-Score':  round(p1_f1,   4),
    'ROC-AUC':   round(p1_auc,  4)
}])
print(df_p1.to_string(index=False))

=== PAPER 1: Binary Risk Detection (LR + TF-IDF) ===
Class distribution: {np.int64(0): np.int64(2524), np.int64(1): np.int64(2476)}
                  Model  Accuracy  Precision  Recall  F1-Score  ROC-AUC
MoodSense (LR + TF-IDF)     0.954     0.9746  0.9313    0.9525   0.9903


In [21]:
print('=== PAPER 2: Wellbeing Scoring (PHQ-9 Validation) ===')

def eval_model(wb_col, label):
    wb           = df[wb_col].values
    phq          = df['phq9_score'].values
    phq_wellness = df['phq9_wellness'].values
    r, p         = spearmanr(wb, -phq)
    mae          = mean_absolute_error(wb, phq_wellness)
    return {
        'Model':             label,
        'Spearman r':        round(r,   3),
        'p-value':           round(p,   4),
        'MAE (0-100 scale)': round(mae, 2)
    }

rows = [
    eval_model('wb_ensemble',   'MoodSense Ensemble'),
    eval_model('wb_bert_only',  'DistilBERT only'),
    eval_model('wb_vader_only', 'VADER only'),
    eval_model('wb_blob_only',  'TextBlob only'),
]

sev_acc_3 = accuracy_score(df['severity_3'], df['pred_3class'])
sev_acc_4 = accuracy_score(df['severity_4'], df['pred_4class'])

df_p2 = pd.DataFrame(rows)
df_p2.loc[df_p2['Model'] == 'MoodSense Ensemble',
          'Severity Acc (3-class)'] = round(sev_acc_3, 3)
df_p2.loc[df_p2['Model'] == 'MoodSense Ensemble',
          'Severity Acc (4-class)'] = round(sev_acc_4, 3)
df_p2 = df_p2.fillna('—')

print(df_p2.to_string(index=False))
print('\n3-class severity report:')
print(classification_report(
    df['severity_3'], df['pred_3class'],
    target_names=['Minimal','Moderate','Severe'],
    zero_division=0
))

=== PAPER 2: Wellbeing Scoring (PHQ-9 Validation) ===
             Model  Spearman r  p-value  MAE (0-100 scale) Severity Acc (3-class) Severity Acc (4-class)
MoodSense Ensemble       0.196      0.0              45.44                  0.516                  0.481
   DistilBERT only       0.064      0.0              60.21                      —                      —
        VADER only       0.287      0.0              38.59                      —                      —
     TextBlob only      -0.058      0.0              26.83                      —                      —

3-class severity report:
              precision    recall  f1-score   support

     Minimal       0.66      0.15      0.25      2524
    Moderate       0.00      0.00      0.00         0
      Severe       0.52      0.89      0.65      2476

    accuracy                           0.52      5000
   macro avg       0.39      0.35      0.30      5000
weighted avg       0.59      0.52      0.45      5000



In [22]:
print('=== Ablation Study ===')

weight_configs = [
    ('BERT only',                       1.00, 0.00, 0.00),
    ('VADER only',                      0.00, 1.00, 0.00),
    ('TextBlob only',                   0.00, 0.00, 1.00),
    ('MoodSense default (0.5/0.3/0.2)', 0.50, 0.30, 0.20),
    ('BERT-heavy (0.6/0.2/0.2)',        0.60, 0.20, 0.20),
    ('Equal (0.33/0.33/0.34)',          0.33, 0.33, 0.34),
    ('VADER-heavy (0.2/0.6/0.2)',       0.20, 0.60, 0.20),
]

ablation_rows = []
for name, w1, w2, w3 in weight_configs:
    wb     = wellbeing_score(df['s_bert'], df['s_vader'], df['s_blob'], w1, w2, w3)
    r, p   = spearmanr(wb, -df['phq9_score'])
    mae    = mean_absolute_error(wb, df['phq9_wellness'])
    pred_3 = wb.apply(wb_to_3class)
    acc    = accuracy_score(df['severity_3'], pred_3)
    ablation_rows.append({
        'Configuration':          name,
        'Weights (B/V/T)':        f'{w1}/{w2}/{w3}',
        'Spearman r':             round(r,   3),
        'MAE':                    round(mae, 2),
        'Severity Acc (3-class)': round(acc, 3)
    })

df_ablation = pd.DataFrame(ablation_rows)
print(df_ablation.to_string(index=False))

=== Ablation Study ===
                  Configuration Weights (B/V/T)  Spearman r   MAE  Severity Acc (3-class)
                      BERT only     1.0/0.0/0.0       0.064 60.21                   0.526
                     VADER only     0.0/1.0/0.0       0.287 38.59                   0.427
                  TextBlob only     0.0/0.0/1.0      -0.058 26.83                   0.091
MoodSense default (0.5/0.3/0.2)     0.5/0.3/0.2       0.224 42.37                   0.445
       BERT-heavy (0.6/0.2/0.2)     0.6/0.2/0.2       0.196 45.44                   0.516
         Equal (0.33/0.33/0.34)  0.33/0.33/0.34       0.196 36.78                   0.381
      VADER-heavy (0.2/0.6/0.2)     0.2/0.6/0.2       0.259 37.15                   0.366


In [23]:
fpr, tpr, _ = roc_curve(y_test, y_prob_lr)

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(fpr, tpr, lw=2, color='#185FA5',
        label=f'LR + TF-IDF (AUC = {p1_auc:.4f})')
ax.plot([0,1],[0,1], '--', color='gray', lw=1, label='Random baseline')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curve — Binary Risk Detection', fontsize=13)
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig('roc_curve.png', dpi=150)
plt.show()
print('Saved: roc_curve.png')

Saved: roc_curve.png


In [24]:
cm     = confusion_matrix(df['severity_3'], df['pred_3class'])
labels = ['Minimal\n(PHQ 0-4)', 'Moderate\n(PHQ 5-14)', 'Severe\n(PHQ 15-27)']

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels, yticklabels=labels,
            ax=ax, linewidths=0.5, linecolor='white')
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('Actual PHQ-9', fontsize=12)
ax.set_title('Confusion Matrix — 3-Class Severity', fontsize=13)
fig.tight_layout()
fig.savefig('confusion_matrix.png', dpi=150)
plt.show()
print('Saved: confusion_matrix.png')

Saved: confusion_matrix.png


In [25]:
fig, ax = plt.subplots(figsize=(6, 5))
scatter = ax.scatter(
    df['phq9_score'], df['wb_ensemble'],
    c=df['severity_3'], cmap='RdYlGn',
    alpha=0.4, s=12, edgecolors='none'
)
r_val, p_val = spearmanr(df['wb_ensemble'], -df['phq9_score'])
ax.set_xlabel('PHQ-9 Score (0–27)', fontsize=12)
ax.set_ylabel('MoodSense Wellbeing Score (0–100)', fontsize=12)
ax.set_title(
    f'Wellbeing Score vs PHQ-9  |  r={r_val:.3f}, p={p_val:.4f}',
    fontsize=11
)
plt.colorbar(scatter, ax=ax, label='Severity (0=min, 2=severe)')
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig('scatter_wb_phq9.png', dpi=150)
plt.show()
print('Saved: scatter_wb_phq9.png')

Saved: scatter_wb_phq9.png


In [26]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
configs = df_ablation['Configuration'].str.split('(').str[0].str.strip()
colors  = ['#185FA5' if 'default' in c.lower() else '#B5D4F4'
           for c in df_ablation['Configuration']]

axes[0].barh(configs, df_ablation['Spearman r'], color=colors)
axes[0].set_xlabel('Spearman r', fontsize=11)
axes[0].set_title('Ablation — Spearman r', fontsize=12)
axes[0].axvline(0, color='gray', lw=0.8, linestyle='--')
axes[0].grid(axis='x', alpha=0.3)

axes[1].barh(configs, df_ablation['Severity Acc (3-class)'], color=colors)
axes[1].set_xlabel('3-Class Severity Accuracy', fontsize=11)
axes[1].set_title('Ablation — Severity Accuracy', fontsize=12)
axes[1].grid(axis='x', alpha=0.3)

fig.suptitle('Fusion Weight Ablation Study', fontsize=13)
fig.tight_layout()
fig.savefig('ablation_chart.png', dpi=150)
plt.show()
print('Saved: ablation_chart.png')

Saved: ablation_chart.png


In [27]:
import shutil

figures = ['roc_curve.png', 'confusion_matrix.png',
           'scatter_wb_phq9.png', 'ablation_chart.png']

for f in figures:
    if os.path.exists(f):
        shutil.copy(f, SAVE_PATH + f)
        print(f'Saved: {f}')

df_p1.to_csv(SAVE_PATH + 'paper1_results.csv', index=False)
df_p2.to_csv(SAVE_PATH + 'paper2_results.csv', index=False)
df_ablation.to_csv(SAVE_PATH + 'ablation_results.csv', index=False)
df[['text','phq9_score','severity_3','wb_ensemble',
    'pred_3class','s_bert','s_vader','s_blob']].to_csv(
    SAVE_PATH + 'full_predictions.csv', index=False
)

print(f'\nAll files saved to: {SAVE_PATH}')

Saved: roc_curve.png
Saved: confusion_matrix.png
Saved: scatter_wb_phq9.png
Saved: ablation_chart.png

All files saved to: /content/drive/MyDrive/MoodSense_Paper2/


In [29]:
print('=' * 60)
print('MOODSENSE PAPER 2 — FINAL RESULTS')
print('=' * 60)
print('\n--- PAPER 1: Binary Risk Detection ---')
print(df_p1.to_string(index=False))
print('\n--- PAPER 2: PHQ-9 Wellbeing Scoring ---')
print(df_p2.to_string(index=False))
print('\n--- Ablation Study ---')
print(df_ablation.to_string(index=False))
print('\nNotes:')
print('  Spearman r: correctly signed (WB vs -PHQ9)')
print('  MAE: PHQ-9 rescaled to 0-100 wellness scale')
print('  Dataset: real clinically-labeled depression data')

MOODSENSE PAPER 2 — FINAL RESULTS

--- PAPER 1: Binary Risk Detection ---
                  Model  Accuracy  Precision  Recall  F1-Score  ROC-AUC
MoodSense (LR + TF-IDF)     0.954     0.9746  0.9313    0.9525   0.9903

--- PAPER 2: PHQ-9 Wellbeing Scoring ---
             Model  Spearman r  p-value  MAE (0-100 scale) Severity Acc (3-class) Severity Acc (4-class)
MoodSense Ensemble       0.196      0.0              45.44                  0.516                  0.481
   DistilBERT only       0.064      0.0              60.21                      —                      —
        VADER only       0.287      0.0              38.59                      —                      —
     TextBlob only      -0.058      0.0              26.83                      —                      —

--- Ablation Study ---
                  Configuration Weights (B/V/T)  Spearman r   MAE  Severity Acc (3-class)
                      BERT only     1.0/0.0/0.0       0.064 60.21                   0.526
           